# Stanford De-Identifier PHI NER

Model: `StanfordAIMI/stanford-deidentifier-base`

This notebook tests Stanford AIMI's de-identification model on a clinical sentence. It is designed for PHI detection, not medication or symptom extraction.

## Supported Entities

| Entity | Meaning |
| --- | --- |
| `DATE` | Dates and date-like expressions |
| `HCW` | Health care worker names, such as doctors or clinical staff |
| `HOSPITAL` | Hospital or care-site names |
| `ID` | Identifiers |
| `PATIENT` | Patient names |
| `PHONE` | Phone numbers |
| `VENDOR` | Vendor or organization names |
| `O` | Outside any entity |

The notebook returns only detected entity spans, so `O` rows are omitted.

In [1]:
from __future__ import annotations

import pandas as pd
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline

MODEL_NAME = "StanfordAIMI/stanford-deidentifier-base"
device = 0 if torch.cuda.is_available() else -1

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

ner_pipeline = pipeline(
    task="token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=device,
)

print(f"Loaded {MODEL_NAME} on {'cuda' if device == 0 else 'cpu'}.")
print("Labels:")
print(model.config.id2label)

tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

C:\Users\vasav\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vasav\.cache\huggingface\hub\models--StanfordAIMI--stanford-deidentifier-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Device set to use cpu


Loaded StanfordAIMI/stanford-deidentifier-base on cpu.
Labels:
{0: 'O', 1: 'VENDOR', 2: 'DATE', 3: 'HCW', 4: 'HOSPITAL', 5: 'ID', 6: 'PATIENT', 7: 'PHONE'}


Edit `input_text`, then run the detection cells.

In [2]:
input_text = "On 03/12/2019 Dr. Smith at St. Mary's Hospital prescribed ibuprofen to Mr. Jones for severe knee pain."
input_text

"On 03/12/2019 Dr. Smith at St. Mary's Hospital prescribed ibuprofen to Mr. Jones for severe knee pain."

In [3]:
def detect_phi_entities(text: str) -> pd.DataFrame:
    predictions = ner_pipeline(text)
    rows = []

    for prediction in predictions:
        rows.append(
            {
                "entity": prediction.get("entity_group", prediction.get("entity")),
                "text": text[prediction["start"]:prediction["end"]],
                "pipeline_text": prediction["word"],
                "start": prediction["start"],
                "end": prediction["end"],
                "score": round(float(prediction["score"]), 4),
            }
        )

    return pd.DataFrame(
        rows,
        columns=["entity", "text", "pipeline_text", "start", "end", "score"],
    )


entities = detect_phi_entities(input_text)
entities

,entity,text,pipeline_text,start,end,score
0,DATE,03/12/2019,03 / 12 / 2019,3,13,0.9980
1,HCW,Dr. Smith,dr. smith,14,23,0.9982
2,HOSPITAL,St. Mary's Hospital,st. mary ' s hospital,27,46,0.9925
3,PATIENT,Mr. Jones,mr. jones,71,80,0.9848


In [4]:
if entities.empty:
    print("No PHI entities detected.")
else:
    for row in entities.itertuples(index=False):
        print(f"{row.entity}: {row.text} ({row.score})")

DATE: 03/12/2019 (0.998)
HCW: Dr. Smith (0.9982)
HOSPITAL: St. Mary's Hospital (0.9925)
PATIENT: Mr. Jones (0.9848)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Expected behavior on the default sentence: this model may detect the date, doctor/staff name, hospital name, and patient name. It should not detect `ibuprofen`, `knee`, or `pain` because those are not part of this model's PHI label space.